# Module 1 — Causal-Chain Hypothesis Sweep
**MADT6004 Wrap-Up · Brew Lab BKK case**

Khun Ploy: *"Revenue is lumpy across branches and weeks. Tell me what's actually true and what we just assumed."*

What you'll do:
1. Identify a **causal chain** as a Python dictionary — every relationship we want to test, with cause, effect, and data types.
2. Write a **dispatch function** that picks the right test (t-test / ANOVA / Pearson / chi-square) based on the data types in each chain.
3. Run the **sweep** across all chains, apply **FDR correction** (Benjamini-Hochberg), and filter to significant findings.
4. Build a **dashboard** of significant relationships.

## 1. Setup

In [1]:
import sqlite3
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

DB_PATH = "../data/brewlab.db"
conn = sqlite3.connect(DB_PATH)
print("Tables:", [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()])

Tables: ['branches', 'customers', 'products', 'promotions', 'transactions', 'order_items', 'loyalty_events', 'campaign_responses', 'reviews']


## 2. Build the analysis frame — daily branch panel

In [2]:
# Aggregate transactions to daily-branch level + join branch attributes
panel = pd.read_sql('''
SELECT date(t.datetime) AS d,
       t.branch_id,
       b.name      AS branch_name,
       b.district,
       b.has_drive_thru,
       b.seats,
       b.size_sqm,
       b.base_traffic,
       SUM(t.total)        AS revenue,
       COUNT(*)            AS orders,
       AVG(t.total)        AS avg_ticket,
       SUM(CASE WHEN t.channel='app' THEN 1 ELSE 0 END)*1.0 / COUNT(*) AS app_share,
       MAX(CASE WHEN t.promo_id IS NOT NULL THEN 1 ELSE 0 END) AS promo_active
FROM transactions t
JOIN branches b ON t.branch_id = b.branch_id
GROUP BY date(t.datetime), t.branch_id
''', conn)
panel['d'] = pd.to_datetime(panel['d'])
panel['is_weekend'] = (panel['d'].dt.weekday >= 5).astype(int)
panel['month']      = panel['d'].dt.to_period('M').astype(str)
panel.head()

,d,branch_id,branch_name,district,has_drive_thru,seats,size_sqm,base_traffic,revenue,orders,avg_ticket,app_share,promo_active,is_weekend,month
0,2025-05-01,1,Asoke,Watthana,0,38,95,1.00,177.95,2,88.975,0.5,0,0,2025-05
1,2025-05-01,3,Thonglor,Watthana,0,45,110,1.05,131.67,1,131.670,0.0,0,0,2025-05
2,2025-05-01,9,Ramkhamhaeng,Bang Kapi,1,28,145,0.80,238.02,1,238.020,0.0,0,0,2025-05
3,2025-05-01,11,Rama 9,Huai Khwang,1,35,155,0.82,70.67,1,70.670,1.0,0,0,2025-05
4,2025-05-02,7,Ari,Phaya Thai,0,36,90,0.95,109.01,1,109.010,0.0,0,0,2025-05


## 3. Define the causal chain as a dictionary

Each entry is a *hypothesized relationship*. We declare:
- `cause` — the explanatory variable
- `effect` — the outcome variable
- `cause_type` / `effect_type` — `binary`, `categorical`, `continuous`
- `subset` — optional filter (e.g., only weekends)
- `direction` — what we expect (for interpretation, not testing)

This dictionary is the *audit trail* of what was tested. No more "I ran 50 tests and reported the p<.05 ones."


In [3]:
CAUSAL_CHAIN = {
    "h01_dt_revenue":        {"cause":"has_drive_thru", "effect":"revenue",     "cause_type":"binary",      "effect_type":"continuous"},
    "h02_dt_weekend_revenue":{"cause":"has_drive_thru", "effect":"revenue",     "cause_type":"binary",      "effect_type":"continuous", "subset":"is_weekend==1"},
    "h03_dt_weekday_revenue":{"cause":"has_drive_thru", "effect":"revenue",     "cause_type":"binary",      "effect_type":"continuous", "subset":"is_weekend==0"},
    "h04_weekend_revenue":   {"cause":"is_weekend",     "effect":"revenue",     "cause_type":"binary",      "effect_type":"continuous"},
    "h05_promo_revenue":     {"cause":"promo_active",   "effect":"revenue",     "cause_type":"binary",      "effect_type":"continuous"},
    "h06_promo_orders":      {"cause":"promo_active",   "effect":"orders",      "cause_type":"binary",      "effect_type":"continuous"},
    "h07_seats_revenue":     {"cause":"seats",          "effect":"revenue",     "cause_type":"continuous",  "effect_type":"continuous"},
    "h08_size_revenue":      {"cause":"size_sqm",       "effect":"revenue",     "cause_type":"continuous",  "effect_type":"continuous"},
    "h09_seats_orders":      {"cause":"seats",          "effect":"orders",      "cause_type":"continuous",  "effect_type":"continuous"},
    "h10_appshare_ticket":   {"cause":"app_share",      "effect":"avg_ticket",  "cause_type":"continuous",  "effect_type":"continuous"},
    "h11_district_revenue":  {"cause":"district",       "effect":"revenue",     "cause_type":"categorical", "effect_type":"continuous"},
    "h12_district_ticket":   {"cause":"district",       "effect":"avg_ticket",  "cause_type":"categorical", "effect_type":"continuous"},
    "h13_dt_avg_ticket":     {"cause":"has_drive_thru", "effect":"avg_ticket",  "cause_type":"binary",      "effect_type":"continuous"},
    "h14_dt_appshare":       {"cause":"has_drive_thru", "effect":"app_share",   "cause_type":"binary",      "effect_type":"continuous"},
    "h15_weekend_ticket":    {"cause":"is_weekend",     "effect":"avg_ticket",  "cause_type":"binary",      "effect_type":"continuous"},
    "h16_promo_ticket":      {"cause":"promo_active",   "effect":"avg_ticket",  "cause_type":"binary",      "effect_type":"continuous"},
    "h17_traffic_revenue":   {"cause":"base_traffic",   "effect":"revenue",     "cause_type":"continuous",  "effect_type":"continuous"},
    "h18_dt_orders_wknd":    {"cause":"has_drive_thru", "effect":"orders",      "cause_type":"binary",      "effect_type":"continuous", "subset":"is_weekend==1"},
    "h19_size_orders":       {"cause":"size_sqm",       "effect":"orders",      "cause_type":"continuous",  "effect_type":"continuous"},
    "h20_appshare_orders":   {"cause":"app_share",      "effect":"orders",      "cause_type":"continuous",  "effect_type":"continuous"},
}
print(f"{len(CAUSAL_CHAIN)} hypotheses queued.")

20 hypotheses queued.


## 4. Dispatch function — pick the right test by data types

| Cause | Effect | Test |
|---|---|---|
| binary | continuous | t-test |
| categorical (k>2) | continuous | One-way ANOVA |
| continuous | continuous | Pearson correlation |
| categorical | categorical | Chi-square |

The function returns a uniform schema (statistic, p-value, direction).

In [ ]:
def run_test(df, h):
    cause, effect = h["cause"], h["effect"]
    ct, et = h["cause_type"], h["effect_type"]
    if "subset" in h:
        df = df.query(h["subset"])

    if ct == "binary" and et == "continuous":
        a = df[df[cause] == 1][effect].dropna()
        b = df[df[cause] == 0][effect].dropna()
        t, p = stats.ttest_ind(a, b)
        return {"test":"t-test", "stat":t, "p":p,
                "summary":f"{a.mean():.0f} vs {b.mean():.0f} (n={len(a)},{len(b)})"}

    if ct == "categorical" and et == "continuous":
        groups = [g[effect].dropna().values for _, g in df.groupby(cause)]
        f, p = stats.f_oneway(*groups)
        return {"test":"ANOVA", "stat":f, "p":p,
                "summary":f"{len(groups)} groups"}

    if ct == "continuous" and et == "continuous":
        sub = df[[cause, effect]].dropna()
        r, p = stats.pearsonr(sub[cause], sub[effect])
        return {"test":"Pearson", "stat":r, "p":p,
                "summary":f"r={r:+.3f} (n={len(sub)})"}

    if ct == "categorical" and et == "categorical":
        ct_table = pd.crosstab(df[cause], df[effect])
        chi2, p, _, _ = stats.chi2_contingency(ct_table)
        return {"test":"chi2", "stat":chi2, "p":p,
                "summary":f"{ct_table.shape[0]}x{ct_table.shape[1]} table"}

    raise ValueError(f"Unsupported types: {ct} -> {et}")

## 5. Sweep all hypotheses + FDR correction

In [5]:
results = []
for hid, h in CAUSAL_CHAIN.items():
    try:
        r = run_test(panel, h)
        results.append({"hypothesis": hid, "cause": h["cause"], "effect": h["effect"],
                        **r})
    except Exception as e:
        print(f"  [skip] {hid}: {e}")

res_df = pd.DataFrame(results)
# Benjamini-Hochberg FDR correction
rejected, p_adj, _, _ = multipletests(res_df["p"].values, alpha=0.05, method="fdr_bh")
res_df["p_adj"]    = p_adj
res_df["significant"] = rejected
res_df = res_df.sort_values("p_adj").reset_index(drop=True)
print(f"Significant after FDR (q=0.05): {res_df.significant.sum()} / {len(res_df)}")
res_df

Significant after FDR (q=0.05): 15 / 20


,hypothesis,cause,effect,test,stat,p,summary,p_adj,significant
0,h12_district_ticket,district,0.353606,ANOVA,338.698693,0.000000e+00,"8 groups, eta²=0.354",0.000000e+00,True
1,h15_weekend_ticket,is_weekend,1.417385,t-test,37.844321,1.140484e-232,"177 vs 130 (n=1234,3108)",1.140484e-231,True
2,h13_dt_avg_ticket,has_drive_thru,-0.944151,t-test,-32.062856,4.855246e-199,"121 vs 155 (n=1449,2893)",3.236830e-198,True
3,h11_district_revenue,district,0.077459,ANOVA,51.984821,1.484730e-71,"8 groups, eta²=0.077",7.423650e-71,True
4,h17_traffic_revenue,base_traffic,0.253590,Pearson,0.253590,1.086653e-64,r=+0.254 (n=4342),4.346611e-64,True
5,h07_seats_revenue,seats,0.227284,Pearson,0.227284,5.478305e-52,r=+0.227 (n=4342),1.826102e-51,True
6,h04_weekend_revenue,is_weekend,0.516499,t-test,13.648361,1.878039e-40,"10836 vs 8084 (n=1234,3108)",5.365826e-40,True
7,h03_dt_weekday_revenue,has_drive_thru,-0.454569,t-test,-13.210957,1.170269e-38,"6650 vs 8803 (n=1039,2069)",2.925674e-38,True
8,h01_dt_revenue,has_drive_thru,-0.375628,t-test,-12.619982,9.415405e-36,"7517 vs 9541 (n=1449,2893)",2.092312e-35,True
9,h06_promo_orders,promo_active,0.267786,t-test,8.651598,7.374507e-18,"66 vs 57 (n=2328,2014)",1.474901e-17,True


## 6. Confirmatory regression with interaction

The naive H02 says drive-thru weekend revenue is *lower* (because drive-thru branches sit in lower-prestige districts). But the *weekend lift inside* drive-thru branches may still be larger. This is the kind of finding the sweep alone hides — a confirmatory regression with the right interaction reveals it.


In [6]:
# Daily revenue ~ has_drive_thru * is_weekend + controls
model = smf.ols(
    "revenue ~ has_drive_thru * is_weekend + seats + size_sqm + base_traffic + C(district) + promo_active",
    data=panel
).fit()
print(model.summary().tables[1])
print("\nKey interaction: has_drive_thru:is_weekend  →  the *additional* weekend lift drive-thru branches enjoy on top of non-DT.")

                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept                   4159.8589   1582.092      2.629      0.009    1058.149    7261.569
C(district)[T.Bang Na]      1225.8649    518.983      2.362      0.018     208.393    2243.337
C(district)[T.Bang Rak]     1515.0890    555.431      2.728      0.006     426.160    2604.018
C(district)[T.Huai Khwang]  1696.9760    348.155      4.874      0.000    1014.414    2379.538
C(district)[T.Khlong Toei]  -971.7950    325.892     -2.982      0.003   -1610.711    -332.879
C(district)[T.Phaya Thai]     46.1971    491.582      0.094      0.925    -917.555    1009.949
C(district)[T.Sathon]        597.5240    601.820      0.993      0.321    -582.352    1777.400
C(district)[T.Watthana]     1039.3568    452.306      2.298      0.022     152.605    1926.109
has_drive_thru              1933.4870   2936.206  

## 7. Dashboard — significant findings only

In [ ]:
sig = res_df[res_df.significant].copy()
sig["neg_log_q"] = -np.log10(sig["p_adj"].clip(lower=1e-300))
sig = sig.sort_values("neg_log_q")

fig, ax = plt.subplots(figsize=(10, max(4, 0.45*len(sig))))
ax.barh(sig["hypothesis"], sig["neg_log_q"], color="#0891B2")
ax.axvline(-np.log10(0.05), color="#DC2626", lw=0.8, ls="--", label="q = 0.05")
ax.set_xlabel("-log10(q)")
ax.set_title("Significant Relationships (FDR q < 0.05)", fontsize=14, fontweight="bold")
for y, (q, summary) in enumerate(zip(sig["p_adj"], sig["summary"])):
    ax.text(sig["neg_log_q"].iloc[y], y, f"  q={q:.3g}  {summary}",
            va="center", fontsize=8, color="#374151")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 8. Discussion prompts

1. Which hypotheses survived FDR — and which intuitive ones *failed* to survive?
2. What does the regression interaction tell us that the pairwise sweep didn't?
3. If you had to pick **three** drivers of revenue to optimize, which would you choose, and why?
4. What confounders are still uncontrolled? How would you address them in a follow-up analysis?

**Next:** Module 2 — forecasting SKU × Branch demand for the next 30 days.
